# Nutrition5k: resultados de MobileNet

No entrena nada: lee los runs de `outputs/mobilenet/` (los escribe `experiments.ipynb` de esta carpeta) con
`src/nutrition5k/results.py`. Se elige por **validación** (`val_mean_pmae`, PMAE medio de los 5 objetivos, menor es
mejor); test solo se informa. Diferencias de menos de ~1 punto son empate (ruido entre semillas).

In [ ]:
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod: the clone made by experiments.ipynb
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().parent.name == "notebooks":
    os.chdir("../..")

import pandas as pd

from nutrition5k.results import (
    best_heads, depth_table, extras_table, head_chart, head_table, load_heads, load_runs, loss_curves,
    overfitting_table, stage2_runs, target_errors,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

FAMILY = "mobilenet"
SPLIT = "full"  # "full" o "pilot"

heads = load_heads([FAMILY], SPLIT)
assert len(heads), f"no stage-1 heads of {FAMILY} in {SPLIT}; run them with experiments.ipynb"
print(f"{len(heads)} heads of {heads['backbone'].nunique()} backbones")

## Etapa 1: cabezas con el backbone congelado

Nombre de cada cabeza: `trunk · cabeza`. El trunk es la parte compartida (`0` = ninguna, `512`, `1024x1024`); la
cabeza es `linear` (una capa lineal a los 5 objetivos), `single 256` (una capa oculta de 256 para los 5) o
`grouped 256` (una capa oculta de 256 por grupo de objetivos). La cabeza por defecto es `512 · single 256`.
Semilla 0, flip horizontal, 80 épocas, lr 1e-3.

### PMAE de validación por cabeza y backbone

En verde, la mejor cabeza de cada backbone; `mean` es la media de los backbones de la familia (la raya negra en la
gráfica).

In [ ]:
display(head_table(heads))
head_chart(heads, f"{FAMILY}: validation mean PMAE per head")

### Mejor cabeza de cada backbone frente a la cabeza por defecto

`default - best` es lo que se pierde por usar la cabeza por defecto. Compáralo con la SD entre semillas de la cabeza
por defecto (semillas 0, 1 y 2): si es menor, la diferencia es ruido.

In [ ]:
best_heads(heads)

### Augmentación y pooling (cabeza por defecto, semilla 0)

`flip` = flip horizontal; `d4` = las 8 simetrías del cuadrado (giros de 90° y flips). En los ViT puros, `flip` y `d4`
usan el token CLS; `mean` (media de los tokens de patch) y `cls+mean` (los dos concatenados) usan flip.
Negativo en `d4 - flip` = D4 mejora.

In [ ]:
extras_table(heads)

### PMAE de validación por objetivo

La cabeza por defecto (`default`) y la mejor de la rejilla (`best`) de cada backbone.

In [ ]:
target_errors(heads)

## Etapa 2: profundidad del fine-tuning

Misma receta para todos: cabeza `512 · single 256`, D4, lr inicial 1e-4 (backbone) y 1e-3 (cabeza) con coseno hasta 0,
semilla 0. Niveles: `frozen` (la cabeza sobre las features guardadas), `low` y `high` (1 y 2 stages) y `all`.

### PMAE de validación por backbone y nivel

En verde, el mejor nivel de cada backbone. Menos de ~1 punto es empate: entonces gana el que entrena menos.

In [ ]:
stage = stage2_runs(load_runs([FAMILY], SPLIT), heads)
depth_table(stage)

### Overfitting y coste de entrenamiento

- `trainable_pct`: % de los parámetros del backbone que se entrenan.
- `val_last5` y `late_rise`: PMAE medio de las 5 últimas épocas y cuánto queda por encima de la mejor época. Con el
  lr en coseno debería ser pequeño; si es grande, la validación empeora al final.
- `train_loss_best`, `val_loss_best` y `gap`: pérdidas (L1 de los objetivos escalados) en la mejor época y su
  diferencia. Un `gap` grande indica que el modelo memoriza train. La de train es la media durante la época con D4,
  así que es algo pesimista.
- `sec_per_epoch` y `gpu_memory_gb`: coste del entrenamiento, comparable solo dentro de un mismo pod.

In [ ]:
overfitting_table(stage)

### Curvas de train y validación

Una gráfica por backbone, un color por nivel (discontinua: train). Hay overfitting si la de train sigue bajando
mientras la de validación se aplana o sube.

In [ ]:
for backbone in stage["backbone"].unique():
    loss_curves(stage[stage["backbone"] == backbone], "unfreeze", f"{backbone}: train vs validation loss")